# Impact of field discrepancy on the classification

In [1]:
from typing import Callable
from types import SimpleNamespace
from copy import deepcopy
from pipe import select

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy import linalg

import plotly.express as px
import plotly.graph_objects as go

In [2]:
SEED = 123
np.random.seed(SEED)

We will take unstable node-type of dynamical system to illustrate why it is important to account for the error in the field function in the trajectory classification.

In [3]:
# system params
systems = [
    {
        "lambda1": 1.,
        "lambda2": 1.6,
        "e1": np.array([1., 0]),
        "e2": np.array([0., 1.]),
    },
    {
        "lambda1": 1.,
        "lambda2": 1.,
        "e1": np.array([1., 0]),
        "e2": np.array([0., 1.]),
    }
]
systems = list(systems | select(lambda d: SimpleNamespace(**d)))

In [4]:
def transition_matrix(sys_c: SimpleNamespace):
    e1, e2 = list(
        [sys_c.e1, sys_c.e2] |
        select(lambda e: e / linalg.norm(e)) |
        select(lambda e: e[:, None])
    )
    U = np.hstack([e1, e2])
    return U

# building matrix A in our ODE x' = A @ x
def system_matrix(
    sys_c: SimpleNamespace
):
    L = np.diag([sys_c.lambda1, sys_c.lambda2])
    U = transition_matrix(sys_c)
    A = (U @ L @ U.T)
    return A

In [5]:
def compute_systems_matrices():
    sys_matrices = list(systems | select(lambda d: system_matrix(d)))
    print(*sys_matrices, sep='\n')
compute_systems_matrices()

[[1.  0. ]
 [0.  1.6]]
[[1. 0.]
 [0. 1.]]


In [6]:
# exact solution
def exact_solver(t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray):
    if t.size == 1:
        return x0[None, :]

    U = transition_matrix(sys_c)
    x0_trans = U.T @ x0
    l = np.repeat(np.array([sys_c.lambda1, sys_c.lambda2])[:, None], t.shape[0], axis=1)
    sol = x0_trans * np.exp(l * t).T
    sol = U @ sol.T
    return sol.T

In [7]:
# numerical solution
def RK_solver(t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray):
    if t.size == 1:
        return x0[None, :]

    A = system_matrix(sys_c)
    return solve_ivp(
        lambda t, x: A.dot(x),
        [t[0], t[-1]],
        x0,
        method="RK23",
        t_eval=t
    ).y.T

In [8]:
x0 = np.array([50., 50.])

In [9]:
T = 1.5
delta_t = 0.5e-1
t = np.linspace(0., T, int(T / delta_t))
t.shape

(30,)

In [10]:
def vizualize_solutions():
    # compute exact solutions
    true_sol = list(
        systems |
        select(lambda s: exact_solver(t, s, x0))
    )
    solutions = {
        "true": true_sol,
    }

    fig = go.Figure()
    for sol_name, sol in solutions.items():
        for i, sys_sol in enumerate(sol):
            line = dict(dash="dash") if sol_name.count("true") == 0 else None
            fig.add_trace(
                go.Scatter(
                    x=sys_sol[:, 0],
                    y=sys_sol[:, 1],
                    mode='lines',
                    name=f"{sol_name}_{i}",
                    line=line
                )
            )
    fig.add_trace(
        go.Scatter(
            x=[x0[0]],
            y=[x0[1]],
            name="start"
        )
    )
    fig.show()

vizualize_solutions()

## Approximate fields

Imagine we have the system 0 with some error: $\lambda_2$ is known with precision of $0.4$. The system 1 is known as it is.

In [11]:
# approximate system params
approx_systems = [
    {
        "lambda1": 1.,
        "lambda2": 2.,
        "e1": np.array([1., 0]),
        "e2": np.array([0., 1.]),
    },
    # system 1 is known exactly
    {
        "lambda1": 1.,
        "lambda2": 1.,
        "e1": np.array([1., 0]),
        "e2": np.array([0., 1.]),
    }
]
approx_systems = list(approx_systems | select(lambda d: SimpleNamespace(**d)))

Let's solve for the true and approximate systems.

In [12]:
def vizualize_solutions():
    # compute exact solutions
    true_sol = list(
        systems |
        select(lambda s: exact_solver(t, s, x0))
    )
    # compute exact solutions for approximate fields
    approx_sol = list(
        approx_systems |
        select(lambda s: exact_solver(t, s, x0))
    )
    # solve approximate sytems with RK23
    approx_RK23_sol = list(
        approx_systems |
        select(lambda s: RK_solver(t, s, x0))
    )
    solutions = {
        "true": true_sol,
        "approx": approx_sol,
        "RK23_approx": approx_RK23_sol
    }

    fig = go.Figure()
    for sol_name, sol in solutions.items():
        for i, sys_sol in enumerate(sol):
            if sol_name == "true":
                line = dict(dash="solid")
            elif sol_name == "approx":
                line = dict(dash="dash")
            else:
                line = dict(dash="longdash")
            fig.add_trace(
                go.Scatter(
                    x=sys_sol[:, 0],
                    y=sys_sol[:, 1],
                    mode='lines',
                    name=f"{sol_name}_{i}",
                    line=line
                )
            )
    fig.add_trace(
        go.Scatter(
            x=[x0[0]],
            y=[x0[1]],
            name="start"
        )
    )
    fig.show()

vizualize_solutions()

## Classification

Let's classify noisy trajectory from system 0 and measure accuracy

In [13]:
# we choose the model with the least mse
def cls_loss(sample_traj: np.ndarray, mean_traj: np.ndarray):
    return np.linalg.norm(sample_traj - mean_traj, ord=2, axis=1).mean()

In [14]:
NUM_SAMPLES = 1000
sigma = 0.01

In [15]:
def simple_classification():
    np.random.seed(SEED)
    accuracy_df = pd.DataFrame(
        {
            "accuracy": np.zeros([3]),
            "method": ["true", "appox", "RK23_approx"],
        }
    )
    traj_0 = exact_solver(t, systems[0], x0)
    num_traj_points = traj_0.shape[0]

    # compute exact solutions
    true_sol = list(
        systems |
        select(lambda s: exact_solver(t, s, x0))
    )
    # compute exact solutions for approximate fields
    approx_sol = list(
        approx_systems |
        select(lambda s: exact_solver(t, s, x0))
    )
    # solve approximate sytems with RK23
    approx_RK23_sol = list(
        approx_systems |
        select(lambda s: RK_solver(t, s, x0))
    )
    solutions = {
        "true": true_sol,
        "approx": approx_sol,
        "RK23_approx": approx_RK23_sol
    }
    
    for i in range(NUM_SAMPLES):
        sample_traj = traj_0 + sigma * np.random.randn(num_traj_points, 2)

        if i == 0:
            fig = go.Figure()
            for sol_name, sol in solutions.items():
                for i, sys_sol in enumerate(sol):
                    # skip approx solutions for system 1 as it is the same as true sys 1
                    if sol_name != "true" and i == 1:
                        continue

                    if sol_name == "true":
                        line = dict(dash="solid")
                    elif sol_name == "approx":
                        line = dict(dash="dash")
                    else:
                        line = dict(dash="longdash")
                    fig.add_trace(
                        go.Scatter(
                            x=sys_sol[:, 0],
                            y=sys_sol[:, 1],
                            mode='lines',
                            name=f"{sol_name}_{i}",
                            line=line
                        )
                    )
            fig.add_trace(
                go.Scatter(
                    x=sample_traj[:, 0],
                    y=sample_traj[:, 1],
                    name="samples",
                    mode='lines+markers',
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[x0[0]],
                    y=[x0[1]],
                    name="start"
                )
            )
            fig.show()

        for sol_name, sol in solutions.items():
            sol_losses = list(
                range(2) |
                select(lambda sys_id: cls_loss(sample_traj, sol[sys_id]))
            )
            if sol_losses[0] < sol_losses[1]:
                accuracy_df.loc[
                    accuracy_df["method"] == sol_name, 
                    "accuracy"
                ] += 1.
    accuracy_df.loc[:, "accuracy"] /= NUM_SAMPLES

    return accuracy_df

accuracy_df = simple_classification()
accuracy_df

,accuracy,method
0,1.0,true
1,0.0,appox
2,0.0,RK23_approx


## Segmented classification

In [16]:
accuracy_df["segment_size"] = t.size

In [17]:
def segmented_solution(
    t: np.ndarray, sys_c: SimpleNamespace, traj: np.ndarray, 
    segment_size: int, solver_f: Callable
):
    """The trajectory is split into segments of length segment_size.
        Each segment is integrated separetely. The overall loss is a sum of
        the losses over the segments. The initial point of each segment is taken
        as it is, we do not account for small noise here.
    """
    sol = []
    cur_start = 0
    while cur_start < t.size:
        cur_x0 = traj[cur_start]
        sol.append(
            solver_f(
                # x0 is included in the segment length
                t[cur_start: min(cur_start + segment_size, t.size)] - t[cur_start],
                sys_c,
                cur_x0
            )
        )
        cur_start = cur_start + segment_size

    return np.concat(sol)

In [18]:
segment_sizes = np.array([10, 5, 3, 1])

## Vizualize the results

In [20]:
fig = px.bar(
    accuracy_df,
    x="segment_size",
    y="accuracy",
    color="method",
    barmode='group',
)
fig.update_xaxes(tickvals=segment_sizes.tolist() + [t.size])
fig.show()